# UD5 - Dashboard interactivo con Dash

En este notebook vamos a construir un dashboard paso a paso. Usaremos los mismos datos de **StreamFlow** que ya conocemos, y los gráficos de Plotly que ya sabemos hacer.

La diferencia: ahora los gráficos van a estar en una **página web** con filtros, y el usuario podrá interactuar con ellos sin tocar código.

### Antes de empezar: ¿qué preguntas debe responder nuestro dashboard?

Antes de escribir una sola línea de código, pensemos qué necesita ver el director de StreamFlow en su pantalla:

1. ¿Cómo van los ingresos este año? (tendencia)
2. ¿Cuántos usuarios activos tenemos? (KPI)
3. ¿Estamos perdiendo usuarios? (churn)
4. ¿De qué países vienen nuestros usuarios? (composición)
5. ¿Cómo se distribuyen por plan? ¿Y varía por país? (filtrable)

Cada pregunta = al menos un elemento visual en el dashboard.

## 0. Instalación e imports

In [1]:
# Ejecuta esto solo la primera vez
!pip install dash --quiet

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Componentes de Dash
from dash import Dash, html, dcc, callback, Output, Input

## 1. Los datos de StreamFlow (los mismos de siempre)

In [3]:
# DATASET 1: Datos mensuales
datos_mensuales = pd.DataFrame({
    'mes': ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
            'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'],
    'mes_num': range(1, 13),
    'mau': [1200000, 1250000, 1310000, 1350000, 1380000, 1340000,
            1290000, 1260000, 1350000, 1420000, 1500000, 1580000],
    'ingresos_k': [320, 335, 352, 360, 370, 355,
                   338, 325, 358, 385, 410, 468],
    'horas_escucha': [1.8, 1.9, 2.0, 1.9, 1.8, 1.7,
                      1.6, 1.5, 1.8, 2.0, 2.1, 2.3],
    'churn_pct': [3.2, 3.0, 2.8, 2.9, 3.1, 3.5,
                  3.8, 4.0, 3.3, 2.7, 2.5, 2.3],
    'conversion_pct': [4.5, 4.7, 5.0, 4.8, 4.6, 4.3,
                       4.1, 3.9, 4.5, 5.1, 5.5, 6.2],
})

# DATASET 2: Usuarios individuales
np.random.seed(42)
n_usuarios = 500

usuarios = pd.DataFrame({
    'plan': np.random.choice(['Free', 'Premium', 'Familiar', 'Estudiante'], 
                              n_usuarios, p=[0.40, 0.30, 0.15, 0.15]),
    'edad': np.random.randint(16, 65, n_usuarios),
    'horas_mes': np.clip(np.random.normal(45, 20, n_usuarios), 5, 150).round(1),
    'pais': np.random.choice(
        ['España', 'México', 'Argentina', 'Colombia', 'Chile', 'Perú'],
        n_usuarios, p=[0.30, 0.25, 0.15, 0.13, 0.10, 0.07]),
    'playlists_creadas': np.random.poisson(5, n_usuarios),
})

usuarios.loc[usuarios['plan'] == 'Premium', 'horas_mes'] *= 1.6
usuarios.loc[usuarios['plan'] == 'Free', 'horas_mes'] *= 0.7
usuarios['horas_mes'] = usuarios['horas_mes'].round(1)
usuarios['gasto_mensual'] = usuarios['plan'].map({
    'Free': 0, 'Premium': 9.99, 'Familiar': 14.99, 'Estudiante': 4.99
})

print(f'Datos mensuales: {len(datos_mensuales)} filas')
print(f'Usuarios: {len(usuarios)} filas')

Datos mensuales: 12 filas
Usuarios: 500 filas


## 1b. Estilos y datos auxiliares

Definimos aquí todos los estilos CSS y las listas que usaremos en los dashboards.
Así no tenemos que volver a definirlos en cada celda.

In [4]:
# =============================================
# ESTILOS (se usan en todos los dashboards)
# =============================================

# --- Contenedor principal de la página ---
estilo_pagina = {
    'fontFamily': 'Arial, sans-serif',
    'maxWidth': '1200px',
    'margin': '0 auto',
    'padding': '20px',
}

# --- Título y subtítulo ---
estilo_titulo = {'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': '5px'}
estilo_subtitulo = {'textAlign': 'center', 'color': '#7f8c8d', 'marginTop': '0'}
estilo_seccion = {'color': '#2c3e50'}

# --- Tarjetas de KPI ---
estilo_tarjeta = {
    'backgroundColor': '#f8f9fa',
    'borderRadius': '8px',
    'padding': '20px',
    'textAlign': 'center',
    'flex': '1',
    'margin': '0 10px',
    'boxShadow': '0 1px 3px rgba(0,0,0,0.1)',
}
estilo_numero = {'fontSize': '36px', 'fontWeight': 'bold', 'color': '#2c3e50', 'margin': '0'}
estilo_etiqueta = {'fontSize': '14px', 'color': '#7f8c8d', 'margin': '5px 0 0 0'}

# --- Layout: filas y columnas (flexbox) ---
estilo_fila = {'display': 'flex', 'marginBottom': '20px'}
estilo_fila_filtros = {'display': 'flex', 'marginBottom': '15px'}
estilo_columna = {'flex': '1'}
estilo_columna_margen = {'flex': '1', 'marginRight': '20px'}

# --- Filtros ---
estilo_label = {'fontWeight': 'bold'}
estilo_fila_label = {'display': 'flex', 'alignItems': 'center', 'marginBottom': '20px'}

# =============================================
# LISTAS PARA FILTROS
# =============================================
paises = ['Todos'] + sorted(usuarios['pais'].unique().tolist())
planes = ['Todos'] + sorted(usuarios['plan'].unique().tolist())
edad = ['Todos'] + sorted(usuarios['edad'].unique().tolist())

# =============================================
# FUNCIÓN AUXILIAR DE FILTRADO
# =============================================
def filtrar(pais, plan):
    datos = usuarios.copy()
    if pais != 'Todos':
        datos = datos[datos['pais'] == pais]
    if plan != 'Todos':
        datos = datos[datos['plan'] == plan]
    return datos

print('Estilos, listas y funciones cargados')

Estilos, listas y funciones cargados


In [5]:
#Crear la aplicación
app = Dash(__name__)

#Gráfico sencillo
fig_ingresos = px.line(
    datos_mensuales,
    x='mes',
    y='ingresos_k',
    title="Evolución de ingresos mensuales",
    markers=True,
    labels={'ingresos_k':'Ingresos','mes':'Mes'},
)

app.layout = html.Div([
    html.H1('Cuadro de mandos de StremFlow'),
    html.P('Panel de métricas del 2025'),
    dcc.Graph(figure=fig_ingresos),
])

app.run(host='0.0.0.0',port=8050, jupyter_mode='external')
print("http://localhost:8050")


Dash app running on http://0.0.0.0:8050/
http://localhost:8050


In [6]:
#Añadimos KPIs
#Crear la aplicación
app = Dash(__name__)

#Gráfico sencillo
fig_ingresos = px.line(
    datos_mensuales,
    x='mes',
    y='ingresos_k',
    title="Evolución de ingresos mensuales",
    markers=True,
    labels={'ingresos_k':'Ingresos','mes':'Mes'},
)

fig_churn = px.line(
    datos_mensuales,
    x='mes',
    y='churn_pct',
    title="Tasa de churn (%)",
    markers=True,
    labels={'churn_pct':'Churn','mes':'Mes'},
    color_discrete_sequence = ['orange']
)

app.layout = html.Div([
    html.H1('Cuadro de mandos de StremFlow', style = estilo_titulo),
    html.P('Panel de métricas del 2025', style = estilo_subtitulo),
    html.Hr(),

    #Fila con las KPIs
    html.Div([
        html.Div([
            html.P(f'{datos_mensuales["mau"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Usuarios activos (MAU)', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{datos_mensuales["ingresos_k"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Inrgresos último mes', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{datos_mensuales["churn_pct"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Tasa de churn', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{len(usuarios)}', style = estilo_numero),
            html.P('Usuarios', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
    ], style = estilo_fila),

    #Dila de gráficas, tendrá dos columnas
    html.Div([
        html.Div([dcc.Graph(figure=fig_ingresos)], style= estilo_columna),
        html.Div([dcc.Graph(figure=fig_churn)],style= estilo_columna),
    ], style = estilo_fila),

], style = estilo_pagina)

app.run(host='0.0.0.0',port=8051, jupyter_mode='external')
print("http://localhost:8051")


Dash app running on http://0.0.0.0:8051/
http://localhost:8051


In [7]:
#Añadimos callbacks

#Añadimos KPIs
#Crear la aplicación
app = Dash(__name__)

#Gráfico sencillo
fig_ingresos = px.line(
    datos_mensuales,
    x='mes',
    y='ingresos_k',
    title="Evolución de ingresos mensuales",
    markers=True,
    labels={'ingresos_k':'Ingresos','mes':'Mes'},
)

fig_churn = px.line(
    datos_mensuales,
    x='mes',
    y='churn_pct',
    title="Tasa de churn (%)",
    markers=True,
    labels={'churn_pct':'Churn','mes':'Mes'},
    color_discrete_sequence = ['orange']
)

app.layout = html.Div([
    html.H1('Cuadro de mandos de StremFlow', style = estilo_titulo),
    html.P('Panel de métricas del 2025', style = estilo_subtitulo),
    html.Hr(),

    #Fila con las KPIs
    html.Div([
        html.Div([
            html.P(f'{datos_mensuales["mau"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Usuarios activos (MAU)', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{datos_mensuales["ingresos_k"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Inrgresos último mes', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{datos_mensuales["churn_pct"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Tasa de churn', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{len(usuarios)}', style = estilo_numero),
            html.P('Usuarios', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
    ], style = estilo_fila),

    #Fila de gráficas FIJAS, tendrá dos columnas
    html.Div([
        html.Div([dcc.Graph(figure=fig_ingresos)], style= estilo_columna),
        html.Div([dcc.Graph(figure=fig_churn)],style= estilo_columna),
    ], style = estilo_fila),

    html.Hr(),

    #Sección dinámica
    html.H2('Análisis por país', style=estilo_seccion),

    #Desplegrable
    html.Div([
        html.Label('Selecciona un país:', style=estilo_label),
        dcc.Dropdown(
            id='filtro-pais',
            options=paises,
            value='Todos',
        )
    ]),
    #Gráficas que sí cambian con el filtro
    html.Div([
        html.Div([dcc.Graph(id='grafico-plan')], style= estilo_columna),
        html.Div([dcc.Graph(id='grafico-horas')],style= estilo_columna),
    ], style = estilo_fila),
    
], style = estilo_pagina)

# CALLBACKS
@callback(
    Output('grafico-plan','figure'),
    Input('filtro-pais','value'),
)
def actualizar_plan(pais):
    if pais == 'Todos':
        datos = usuarios
    else:
        datos = usuarios[usuarios['pais'] == pais ]

    cuenta = datos['plan'].value_counts().reset_index()
    cuenta.columns = ['plan','cantidad']

    fig = px.bar(
        cuenta,
        x='plan',
        y='cantidad',
        title=f'Distribución por plan - {pais}',
        text='cantidad',
        color='plan',
    )
    fig.update_layout(showlegend=False)
    return fig
    
@callback(
    Output('grafico-horas','figure'),
    Input('filtro-pais','value'),
)
def actualizar_horas(pais):
    if pais == 'Todos':
        datos = usuarios
    else:
        datos = usuarios[usuarios['pais'] == pais ]

    fig = px.box(
        datos,
        x='plan',
        y='horas_mes',
        color='plan',
        title=f'Horas de escucha por plan - {pais}',
        )
    fig.update_layout(showlegend=False)
    return fig

app.run(host='0.0.0.0',port=8052, jupyter_mode='external')
print("http://localhost:8052")


Dash app running on http://0.0.0.0:8052/
http://localhost:8052


---

## Ejercicio

Partiendo del dashboard v3, añade lo siguiente:

**Ejercicio A**: Añade un gráfico de tarta (`px.pie`) que muestre la proporción de usuarios por plan, y que se actualice con el filtro de país. Necesitarás:
1. Un `dcc.Graph(id='grafico-tarta')` en el layout.
2. Un `@callback` que escuche `'filtro-pais'` y devuelva el gráfico a `'grafico-tarta'`.

**Ejercicio B**: Sustituye el dropdown de país por un `dcc.RadioItems` (botones de radio). La sintaxis es casi igual:
```python
dcc.RadioItems(
    id='filtro-pais',
    options=paises,
    value='Todos',
    inline=True,       # Para que se muestren en fila
)
```
¿Necesitas cambiar algo en los callbacks? ¿Por qué?

**Ejercicio C**: Añade un `dcc.Slider` para filtrar usuarios por edad mínima.

In [9]:
#Añadimos callbacks

#Añadimos KPIs
#Crear la aplicación
app = Dash(__name__)

#Gráfico sencillo
fig_ingresos = px.line(
    datos_mensuales,
    x='mes',
    y='ingresos_k',
    title="Evolución de ingresos mensuales",
    markers=True,
    labels={'ingresos_k':'Ingresos','mes':'Mes'},
)

fig_churn = px.line(
    datos_mensuales,
    x='mes',
    y='churn_pct',
    title="Tasa de churn (%)",
    markers=True,
    labels={'churn_pct':'Churn','mes':'Mes'},
    color_discrete_sequence = ['orange']
)

app.layout = html.Div([
    html.H1('Cuadro de mandos de StremFlow', style = estilo_titulo),
    html.P('Panel de métricas del 2025', style = estilo_subtitulo),
    html.Hr(),

    #Fila con las KPIs
    html.Div([
        html.Div([
            html.P(f'{datos_mensuales["mau"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Usuarios activos (MAU)', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{datos_mensuales["ingresos_k"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Inrgresos último mes', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{datos_mensuales["churn_pct"].iloc[-1]:,.0f}', style = estilo_numero),
            html.P('Tasa de churn', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
        html.Div([
            html.P(f'{len(usuarios)}', style = estilo_numero),
            html.P('Usuarios', style = estilo_etiqueta),
        ], style = estilo_tarjeta),
    ], style = estilo_fila),

    #Fila de gráficas FIJAS, tendrá dos columnas
    html.Div([
        html.Div([dcc.Graph(figure=fig_ingresos)], style= estilo_columna),
        html.Div([dcc.Graph(figure=fig_churn)],style= estilo_columna),
    ], style = estilo_fila),

    html.Hr(),

    #Sección dinámica
    html.H2('Análisis por país', style=estilo_seccion),

    #Desplegrable
    html.Div([
        html.Label('Selecciona un país:', style=estilo_label),
        dcc.Dropdown(
            id='filtro-pais',
            options=paises,
            value='Todos',
        )
    ]),
    #Gráficas que sí cambian con el filtro
    html.Div([
        html.Div([dcc.Graph(id='grafico-plan')], style= estilo_columna),
        html.Div([dcc.Graph(id='grafico-horas')],style= estilo_columna),
    ], style = estilo_fila),

    html.Div([
        html.Label('Arrastra para elegir la edad mínima:', style=estilo_label),
        dcc.Slider(
            id='filtro-edad',
            min=usuarios['edad'].min(),
            max=usuarios['edad'].max(),
            step=1,
            value=usuarios['edad'].min(),
        )
    ]),
    
], style = estilo_pagina)

# CALLBACKS
@callback(
    Output('grafico-plan','figure'),
    Input('filtro-pais','value'),
)
def actualizar_plan(pais):
    if pais == 'Todos':
        datos = usuarios
    else:
        datos = usuarios[usuarios['pais'] == pais ]

    cuenta = datos['plan'].value_counts().reset_index()
    cuenta.columns = ['plan','cantidad']

    fig = px.bar(
        cuenta,
        x='plan',
        y='cantidad',
        title=f'Distribución por plan - {pais}',
        text='cantidad',
        color='plan',
    )
    fig.update_layout(showlegend=False)
    return fig
    
@callback(
    Output('grafico-horas','figure'),
    Input('filtro-pais','value'),
)
def actualizar_horas(pais):
    if pais == 'Todos':
        datos = usuarios
    else:
        datos = usuarios[usuarios['pais'] == pais ]

    fig = px.box(
        datos,
        x='plan',
        y='horas_mes',
        color='plan',
        title=f'Horas de escucha por plan - {pais}',
        )
    fig.update_layout(showlegend=False)
    return fig

@callback(
    Output('grafico-plan','figure'),
    Input('filtro-edad','value'),
)
def actualizar_edad(edad):
    
    datos = usuarios[usuarios['edad'] > edad]

    cuenta = datos['edad'].value_counts().reset_index()
    cuenta.columns = ['edad','cantidad']

    fig = px.bar(
        cuenta,
        x='edad',
        y='cantidad',
        title=f'Distribución por edad - Min. {edad} años',
        text='cantidad',
        color='edad',
    )
    fig.update_layout(showlegend=False)
    return fig

app.run(host='0.0.0.0',port=8052, jupyter_mode='external')
print("http://localhost:8052")


Dash app running on http://0.0.0.0:8052/
http://localhost:8052


[2026-04-22 19:16:19,030] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/dash/_get_app.py", line 17, in wrap
    r